In [ ]:
import polars as pl
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt
df = pl.read_csv("merged_dedup.csv")

In [ ]:
yearly_counts=pl_df.group_by("Year").len().sort("Year").filter(pl.col("Year")!=2025)

# Option A: Using Matplotlib (most common and flexible)
plt.figure(figsize=(10, 6))
plt.bar(yearly_counts["Year"], yearly_counts["len"], color='skyblue')
plt.xlabel("Year")
plt.ylabel("Number of Publications")
plt.title("Number of Publications per Year")
plt.xticks(yearly_counts["Year"]) # Ensure all years are shown as ticks
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("Evolution Papers.pdf")
plt.show()


In [ ]:
yearly_counts_with_growth.filter(pl.col("yearly_growth_pct") != 0.0)["yearly_growth_pct"].mean()


In [ ]:
# --- Your existing code to get yearly counts (corrected filter) ---
yearly_counts = pl_df.group_by("Year").len().sort("Year").filter(pl.col("Year") != 2025)

# --- Calculate Yearly Growth ---
yearly_counts_with_growth = yearly_counts.with_columns(
    pl.col("len").shift(1).alias("previous_year_len")
)

yearly_counts_with_growth = yearly_counts_with_growth.with_columns(
    pl.when(pl.col("previous_year_len").is_null() | (pl.col("previous_year_len") == 0))
    .then(0.0) # If previous_year_len is null (first year) or 0, growth is 0 or undefined, set to 0.0
    .otherwise(((pl.col("len") - pl.col("previous_year_len")) / pl.col("previous_year_len") * 100))
    .alias("yearly_growth_pct")
)

# --- Plotting the Yearly Growth with Labels ---

plt.figure(figsize=(12, 7))
bars = plt.bar(yearly_counts_with_growth["Year"], yearly_counts_with_growth["yearly_growth_pct"], color='lightcoral')
plt.xlabel("Year")
plt.ylabel("Yearly Growth (%)")
plt.title("Yearly Growth of Publications (Percentage Change)")
plt.xticks(yearly_counts_with_growth["Year"]) # Ensure all years are shown as ticks
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.axhline(0, color='grey', linewidth=0.8) # Add a horizontal line at 0 for reference

# Add percentage labels on top of the bars
for bar in bars:
    yval = bar.get_height()
    # Adjust position slightly based on whether the bar is positive or negative
    # And format to 1 decimal place
    plt.text(bar.get_x() + bar.get_width()/2, yval + (0.1 if yval >= 0 else -1),
             f'{yval:.1f}%', ha='center', va='bottom' if yval >= 0 else 'top',
             fontsize=9, color='black') # Adjust font size and color as needed

plt.savefig("Yearly Growth_with_labels.pdf")
plt.show()

# --- Plotting both counts and growth on separate subplots, with labels on growth ---
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Plot 1: Yearly Counts (no labels, often too crowded for absolute counts)
axes[0].bar(yearly_counts["Year"], yearly_counts["len"], color='skyblue')
axes[0].set_ylabel("Number of Publications")
axes[0].set_title("Number of Publications per Year")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Plot 2: Yearly Growth (with labels)
bars_growth = axes[1].bar(yearly_counts_with_growth["Year"], yearly_counts_with_growth["yearly_growth_pct"], color='lightcoral')
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Yearly Growth (%)")
axes[1].set_title("Yearly Growth of Publications")
axes[1].axhline(0, color='grey', linewidth=0.8)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels on top of the growth bars for the subplot
for bar in bars_growth:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + (0.1 if yval >= 0 else -1),
                 f'{yval:.1f}%', ha='center', va='bottom' if yval >= 0 else 'top',
                 fontsize=9, color='black')

plt.xticks(yearly_counts["Year"])
plt.tight_layout()
plt.savefig("Publications_and_Growth_with_labels.pdf")
plt.show()